# Verifying the music-dsp feedback (R. Bristow-Johnson, June 2026)

After SampleRateTap was announced on the [music-dsp list](https://lists.columbia.edu/mailman/listinfo/music-dsp),
Robert Bristow-Johnson replied with detailed feedback containing several quantitative
claims about polyphase interpolator design. This notebook checks each claim against
this library's actual filters — the design math below is the same formula-for-formula
port of `include/srt/detail/kaiser.hpp` used by `scripts/book_figures.py`, and every
result is pinned by an assertion so regressions in either the claims or the code fail loudly.

**The claims under test:**

1. With linear inter-phase interpolation, folded-image S/N is ~108 dB at L=256 and
   120+ dB at L=512, improving 12 dB per doubling of L (6 dB for drop-sample).
2. "That's a lotta taps" — T=32 taps per phase suffices instead of our T=48.
3. Putting zeros at integer multiples of fs (convolving the prototype with a
   one-sample rect) usefully deepens the low-order images.
4. Two dot products + one output interpolation beats blending coefficients then one dot.
5. The phase table should have L+1 rows spanning delays T/2−1 … T/2 samples, with the
   first prototype tap zero by symmetry.

Method note: claims 1–3 are measured by *actually fractionally resampling* a sine
through the table (ratio 1 + 200 ppm) and least-squares-fitting the expected tone;
the residual is everything the interpolator got wrong. 19.5 kHz is used as the
near-worst case (interpolation error scales with signal frequency); 997 Hz shows
the mid-band behaviour our headline numbers are measured at.

In [1]:
import sys
import numpy as np

sys.path.insert(0, "../scripts")
from book_figures import design_prototype, kaiser_beta  # kaiser.hpp, ported verbatim

FS = 48000.0

def proto(L, T, pass_hz, stop_hz, atten):
    return design_prototype(L, T, (pass_hz + stop_hz) / FS, kaiser_beta(atten))

def resample_snr(L, T, pass_hz, stop_hz, atten, f0, eps=2e-4, N=150_000, mode="linear"):
    """Fractionally resample a sine using the (L+1)-row table with linear
    coefficient interpolation (== linear interpolation on the oversampled
    prototype grid), LS-fit the expected tone, return residual SNR in dB."""
    h = proto(L, T, pass_hz, stop_hz, atten)      # length L*T, grid 1/L, DC gain L
    n = np.arange(N, dtype=float)
    pos = n * (1.0 + eps) + T + 2
    idx = np.floor(pos).astype(int)
    x = np.sin(2 * np.pi * f0 * np.arange(idx.max() + T + 4) / FS)
    m = idx[:, None] + np.arange(-(T // 2) + 1, T // 2 + 1)[None, :]
    u = pos[:, None] - m                           # kernel argument in (-T/2, T/2]
    g = u * L + (L * T - 1) / 2.0                  # oversampled-prototype index
    if mode == "linear":
        g0 = np.floor(g).astype(int); fr = g - g0
        g0c = np.clip(g0, 0, L * T - 1); g1c = np.clip(g0 + 1, 0, L * T - 1)
        P = (1 - fr) * h[g0c] * (g0 >= 0) + fr * h[g1c] * (g0 + 1 <= L * T - 1)
    else:                                          # drop-sample / nearest phase row
        gn = np.rint(g).astype(int)
        P = h[np.clip(gn, 0, L * T - 1)] * ((gn >= 0) & (gn <= L * T - 1))
    y = (x[m] * P).sum(axis=1)
    t = pos / FS
    A = np.column_stack([np.sin(2 * np.pi * f0 * t), np.cos(2 * np.pi * f0 * t)])
    y, A = y[N // 10:], A[N // 10:]                # drop the fill transient
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    r = y - A @ c
    return 10 * np.log10((A @ c).var() / r.var())

## Claim 1 — the L budget: ~108 dB at 256, 120+ at 512, 12 dB/doubling

A 140 dB / T=80 FIR isolates the phase-interpolation error (the FIR's own stopband
sits well below every floor we are trying to measure).

In [2]:
snr = {}
prev = None
for L in (128, 256, 512, 1024):
    snr[L] = resample_snr(L, 80, 20000, 26000, 140, 19500)
    step = "" if prev is None else f"   (+{snr[L]-prev:.1f} dB per doubling)"
    print(f"L={L:5}: {snr[L]:6.1f} dB{step}")
    prev = snr[L]

drop = {L: resample_snr(L, 80, 20000, 26000, 140, 19500, mode="nearest") for L in (256, 512)}
print(f"drop-sample: L=256 {drop[256]:.1f} dB, L=512 {drop[512]:.1f} dB "
      f"(+{drop[512]-drop[256]:.1f} dB per doubling)")

assert 107.0 < snr[256] < 110.0, "RBJ's ~108 dB at L=256"
assert 119.0 < snr[512] < 123.0, "RBJ's 120+ dB at L=512"
assert 11.0 < snr[512] - snr[256] < 13.0, "12 dB per doubling (linear)"
assert 5.0 < drop[512] - drop[256] < 7.0, "6 dB per doubling (drop-sample)"
print("\nCONFIRMED: all four figures match RBJ's rules of thumb.")

L=  128:   96.6 dB


L=  256:  108.6 dB   (+12.0 dB per doubling)


L=  512:  120.7 dB   (+12.0 dB per doubling)


L= 1024:  132.5 dB   (+11.8 dB per doubling)


drop-sample: L=256 50.8 dB, L=512 56.8 dB (+6.0 dB per doubling)

CONFIRMED: all four figures match RBJ's rules of thumb.


**Verdict: confirmed, remarkably precisely.** His ~108/120+/12-per-doubling figures
reproduce to within half a decibel on our own prototype. Note the frequency dependence:
this floor is for a 19.5 kHz tone; the same L=256 table measures ~144 dB at 997 Hz,
which is why the library's mid-band headline measurements (126–135 dB through the full
converter) coexist with a ~108 dB near-Nyquist interpolation floor — and why
`test_asrc_quality`'s 19.5 kHz threshold sits at 100 dB rather than 110.

## Claim 2 — "I think you can get away with T=32"

T is not spent on phase resolution — it is spent on the FIR's own transition width and
stopband depth. Pair T=32 (whose achievable Kaiser design at our band edges is the
96 dB `fast` preset) with as many phases as you like, and the FIR stopband dominates:

In [3]:
cases = [
    ("fast        L=128 T=32", 128, 32, 18000, 30000,  96),
    ("balanced    L=256 T=48", 256, 48, 20000, 28000, 120),
    ("balanced @ L=512      ", 512, 48, 20000, 28000, 120),
    ("RBJ's T=32 @ L=512    ", 512, 32, 18000, 30000,  96),
    ("transparent L=512 T=80", 512, 80, 20000, 26000, 140),
]
res = {}
for name, L, T, pb, sb, att in cases:
    hi = resample_snr(L, T, pb, sb, att, 19500)
    lo = resample_snr(L, T, pb, sb, att, 997)
    res[name.strip()] = (hi, lo)
    print(f"{name}:  19.5 kHz {hi:6.1f} dB    997 Hz {lo:6.1f} dB")

assert res["RBJ's T=32 @ L=512"][0] < 90, "T=32 tops out near the 96 dB design floor"
assert res["balanced    L=256 T=48"][0] > 105
print("\nREFUTED at the 120 dB spec: quadrupling L cannot rescue a 96 dB-class FIR.")

fast        L=128 T=32:  19.5 kHz   83.9 dB    997 Hz  110.7 dB


balanced    L=256 T=48:  19.5 kHz  108.5 dB    997 Hz  144.3 dB


balanced @ L=512      :  19.5 kHz  118.9 dB    997 Hz  144.3 dB


RBJ's T=32 @ L=512    :  19.5 kHz   84.2 dB    997 Hz  110.7 dB


transparent L=512 T=80:  19.5 kHz  120.7 dB    997 Hz  153.5 dB

REFUTED at the 120 dB spec: quadrupling L cannot rescue a 96 dB-class FIR.


**Verdict: refuted at our specification.** T=32 with L=512 delivers 84 dB at
19.5 kHz — the FIR stopband, not the phase count, is the binding constraint. The
harris length estimate says the same thing analytically: 120 dB across an 8 kHz
transition needs T ≈ 47 (Kaiser) or ≈ 44 (equiripple/Parks–McClellan). T=48 *is*
the 120 dB spec; T=32 *is* our `fast` preset, offered for the 96 dB tier. (RBJ's
own designs pair T=32 with optimal PM/LS prototypes and a less demanding stopband
target — in that budget his claim is fair.)

## Claim 3 — zeros at multiples of fs (convolve the prototype with a rect)

Real technique, real benefit at the low-order images — and a real cost RBJ's optimal
designs compensate but a naive rect-on-Kaiser does not: sinc droop across the passband.

In [4]:
L, T = 256, 48
h1 = proto(L, T, 20000, 28000, 120)
h2 = np.convolve(h1, np.ones(L) / L)     # zeros at k*fs; T -> T+1 taps per phase
report = {}
for tag, hh in (("as shipped", h1), ("rect-convolved", h2)):
    nfft = 1 << 21
    H = np.abs(np.fft.rfft(hh, nfft)) / L
    f = np.arange(H.size) * (L * FS) / nfft
    db = 20 * np.log10(np.maximum(H, 1e-16))
    band = lambda lo, hi: db[(f >= lo) & (f <= hi)].max()
    report[tag] = (band(19990, 20010), band(FS - 20000, FS + 20000), band(2*FS - 20000, 2*FS + 20000))
    print(f"{tag:15}: droop@20k {report[tag][0]:6.2f} dB | "
          f"worst image fs±20k {report[tag][1]:7.1f} dB | 2fs±20k {report[tag][2]:7.1f} dB")

droop = report["rect-convolved"][0]
img_gain = report["as shipped"][1] - report["rect-convolved"][1]
assert droop < -2.0, "sinc droop at the passband edge is material"
assert img_gain > 4.0, "the image-band benefit is also real"
print(f"\nBOTH SIDES REAL: {img_gain:.1f} dB deeper first image, {droop:.2f} dB droop at 20 kHz.")

as shipped     : droop@20k   0.00 dB | worst image fs±20k  -119.1 dB | 2fs±20k  -148.0 dB
rect-convolved : droop@20k  -2.64 dB | worst image fs±20k  -124.7 dB | 2fs±20k  -162.3 dB

BOTH SIDES REAL: 5.6 dB deeper first image, -2.64 dB droop at 20 kHz.


**Verdict: correct but not free.** ~6 dB deeper first-image rejection and ~14 dB at
2·fs, in exchange for **−2.6 dB at 20 kHz**, which violates this library's ±0.01 dB
passband spec by two orders of magnitude. Adopting the idea properly means an optimal
(PM/LS) design with the sinc droop pre-compensated in the passband target — a genuine
future experiment, not a drop-in change. (RBJ's MATLAB designs do exactly this.)

## Claim 4 — "two dot products and one linear interpolation would be quicker"

An operation-count question, and for the multichannel case a measured one. Per output
frame with T taps and N channels:

| approach | per-frame ops |
|---|---|
| blend row once, dot per channel (ours) | T blend + N·T MAC ≈ (N+1)·T |
| two dots per channel, lerp outputs (RBJ) | 2·N·T MAC + N lerp ≈ 2·N·T |

Mono: a wash (2T either way — his form even saves the scratch-row traffic; on
Cortex-M33-class targets our Q15 path already prefers fused/dual-MAC forms).
N ≥ 2: the blend amortizes across channels and ours wins by (N−1)·T ops — which is
not hypothetical: the C1 campaign measured **−36 % stereo and −52 % 8-channel**
wall-clock from exactly this restructuring (`docs/PERFORMANCE.md`, C1), and the C6
channel-parallel path (frame-major, one blended row broadcast across channel lanes)
requires the shared blended row to exist at all.

## Claim 5 — table conventions (L+1 rows, delay span, zero first tap)

No disagreement to test — just verify we implement the same convention he describes.

In [5]:
h = proto(256, 48, 20000, 28000, 120)
gd = (256 * 48 - 1) / (2 * 256)
print(f"h[0] = {h[0]:.3e}   (RBJ's convention: exactly 0 by symmetry)")
print(f"group delay = (L·T−1)/(2L) = {gd:.4f} samples = T/2 − 1/(2L)")
print("rows stored: L+1 (row L = row 0 advanced one input sample) — see PolyphaseFilterBank")
assert abs(h[0]) < 1e-7
assert abs(gd - (24 - 1/512)) < 1e-9
print("\nSAME DESIGN: delay spans T/2−1 … T/2 across the L+1 rows, first tap ≈ 0.")

h[0] = -3.331e-09   (RBJ's convention: exactly 0 by symmetry)
group delay = (L·T−1)/(2L) = 23.9980 samples = T/2 − 1/(2L)
rows stored: L+1 (row L = row 0 advanced one input sample) — see PolyphaseFilterBank

SAME DESIGN: delay spans T/2−1 … T/2 across the L+1 rows, first tap ≈ 0.


## Conclusions

| # | Claim | Verdict |
|---|---|---|
| 1 | ~108 dB @ L=256, 120+ @ L=512, 12 dB/doubling (6 for drop-sample) | **Confirmed** to ±0.5 dB |
| 2 | T=32 suffices | **Refuted at the 120 dB spec** — FIR stopband dominates; T=32 is the 96 dB `fast` tier |
| 3 | Zeros at k·fs via rect convolution | **Real benefit, real cost** — needs droop-precompensated optimal design |
| 4 | Two dots + output lerp | **Mono: wash. Multichannel: ours measured faster** (C1: −36 %/−52 %) |
| 5 | L+1 rows, T/2−1…T/2 delay span, zero first tap | **Same convention**, verified |

Also raised in the thread: per-sample timestamping as an alternative servo observable
(interesting for acquisition speed; our occupancy+μ observable is already continuous and
whole-sample slips are bit-continuous by the extra-row property — `AsrcLock.WholeSampleSlipsAreGlitchFree`),
and Q32.32 absolute indexing vs our Q0.64 deviation-only accumulator (equivalent;
near-unity lets the integer advance carry the 1).